# conv-windowing-2d — faded example 1: Complete the as_strided stride tuple for a stride-1 conv view

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-windowing-2d`. Running the beacon reports progress on the `CNN: 2-D conv windowing` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: 2-D conv windowing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-windowing-2d`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-windowing-2d"
DD_SUBTOPIC = "CNN: 2-D conv windowing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The `(B, IC, OH, OW, KH, KW)` window view needs a 6-element stride tuple. The first two entries copy `(s_b, s_ic)`. The middle pair steps *between* windows and the trailing pair steps *within* a window — for stride-1 both pairs are `(s_h, s_w)`.

## Faded exercise 1

### Faded — finish the stride tuple

Implement `conv2d_windows(x, KH, KW)` returning the stride-1 window view `(B, IC, OH, OW, KH, KW)` that shares storage with `x`. The output shape and `size=` argument are given; you must supply the **`stride=` tuple** passed to `as_strided`.

**Fill in:** The 6-element stride tuple addressing batch, channel, between-window (OH, OW), and within-window (KH, KW) axes.

In [ ]:
def conv2d_windows(x: Tensor, KH: int, KW: int) -> Tensor:
    B, IC, H, W = x.shape
    OH = H - KH + 1
    OW = W - KW + 1
    s_b, s_ic, s_h, s_w = x.stride()
    stride = None  # TODO: the 6-element stride tuple for (B, IC, OH, OW, KH, KW)
    return x.as_strided(size=(B, IC, OH, OW, KH, KW), stride=stride)


import torch.nn.functional as F
from einops import einsum

def _test():
    t.manual_seed(1)
    x = t.randn(2, 3, 9, 7)
    KH, KW = 3, 2
    win = conv2d_windows(x, KH, KW)
    B, IC, H, W = x.shape
    OH, OW = H - KH + 1, W - KW + 1
    assert win.shape == (B, IC, OH, OW, KH, KW), win.shape
    assert win.data_ptr() == x.data_ptr(), 'must be a view, not a copy'
    w = t.randn(4, IC, KH, KW)
    out = einsum(win, w, 'b ic oh ow kh kw, oc ic kh kw -> b oc oh ow')
    ref = F.conv2d(x, w)
    assert t.allclose(out, ref, atol=1e-4), (out - ref).abs().max().item()


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def conv2d_windows(x: Tensor, KH: int, KW: int) -> Tensor:
    B, IC, H, W = x.shape
    OH = H - KH + 1
    OW = W - KW + 1
    s_b, s_ic, s_h, s_w = x.stride()
    stride = (s_b, s_ic, s_h, s_w, s_h, s_w)
    return x.as_strided(size=(B, IC, OH, OW, KH, KW), stride=stride)
```
</details>